# COMP64702 - RAG Culinary Assistant
## Notebook 06: Ablation Study + Extended Evaluation (10 Metrics + Confidence)
### =============================================================================
### This notebook:
####   1. Runs ablation study across 6 system configurations
####   2. Evaluates on 10 metrics per configuration
####   3. Computes confidence scores for every predicted answer
####   4. Produces a complete results table ready for presentation
####   5. Runs statistical significance tests across configurations

### 10 Metrics:
####   Generation : ROUGE-1, ROUGE-2, ROUGE-L, BERTScore F1,
####                METEOR, Answer F1, Exact Match
####   Retrieval  : MRR@5, Recall@5, NDCG@5
####   RAG-specific: Faithfulness, Answer Relevance, Context Precision
####   (we report 10 primary ones in the main table)

#### Confidence: entropy-based token probability confidence score
### =============================================================================
 

In [1]:
import os
import sys
import json
import pickle
import time
import re
import numpy as np
import faiss
import torch
from datetime import datetime
from collections import defaultdict
from scipy import stats
from dotenv import load_dotenv
from huggingface_hub import login
 
# Path setup
BASE_DIR = r"D:\text mining\rag project"
os.chdir(BASE_DIR)
sys.path.insert(0, os.path.join(BASE_DIR, "src"))
 
from embedder  import Embedder
from retriever import Retriever
from generator import Generator, build_context
 
# Metric libraries
from rouge_score import rouge_scorer
from bert_score  import score as bert_score_fn
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
 
# NLTK for METEOR
import nltk
nltk.download("wordnet",          quiet=True)
nltk.download("punkt",            quiet=True)
nltk.download("punkt_tab",        quiet=True)
nltk.download("omw-1.4",          quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
from nltk.translate.meteor_score import meteor_score
 
os.makedirs("outputs/ablation", exist_ok=True)
 
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
print("Setup complete")

Setup complete


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Load all models and data
# ─────────────────────────────────────────────────────────────────────────────
 
print("Loading models...")
 
embedder_  = Embedder("BAAI/bge-small-en-v1.5")
retriever_ = Retriever("vector_store", embedder=embedder_)
generator_ = Generator("Qwen/Qwen2.5-0.5B-Instruct")
 
device = "cuda" if torch.cuda.is_available() else "cpu"
 
# Load train benchmark
with open("data/benchmark/train_set.json", encoding="utf-8") as f:
    train_set = json.load(f)
 
# Load existing train outputs (from notebook 04)
with open("outputs/train_outputs.json", encoding="utf-8") as f:
    train_outputs = json.load(f)
 
# Build source lookup: question -> gold source title
source_lookup = {qa["question"]: qa.get("source_title", "") for qa in train_set}
 
print(f"Loaded {len(train_set)} benchmark pairs")
print(f"Loaded {len(train_outputs)} existing predictions")
print("All models ready\n")
 

Loading models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded: BAAI/bge-small-en-v1.5 (dim=384)
FAISS index loaded — 3,356 vectors
Chunks loaded     — 3,356 chunks
BM25 index loaded


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded
Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Generator loaded on cpu
Loaded 92 benchmark pairs
Loaded 92 existing predictions
All models ready



In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Confidence scoring
# Measures how confident the model is in its answer using token probabilities.
# High entropy = uncertain / low confidence
# Low entropy  = certain  / high confidence
# ─────────────────────────────────────────────────────────────────────────────
 
def compute_confidence(query, context, answer,
                       system_prompt, user_prompt_fn,
                       tokenizer, model, device, max_new_tokens=200):
    """
    Computes a confidence score for a generated answer using token probability.
 
    Method:
        1. Run the same prompt through the model
        2. Collect softmax probabilities for each generated token
        3. Compute mean token probability (higher = more confident)
        4. Compute entropy (lower = more confident)
        5. Return normalised confidence score in [0, 1]
 
    Args:
        query          : the question
        context        : retrieved context string
        answer         : previously generated answer string
        system_prompt  : system message
        user_prompt_fn : function(query, context) -> user message string
        tokenizer      : Qwen tokenizer
        model          : Qwen model
        device         : cpu or cuda
        max_new_tokens : generation length
 
    Returns:
        dict with mean_prob, entropy, confidence (0-1), token_count
    """
    from src.generator import STRATEGIES, build_context
 
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt_fn(query, context)},
    ]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
 
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,          # ← collect token scores
            repetition_penalty=1.1,
        )
 
    # Extract token probabilities from scores
    token_probs = []
    for step_scores in output.scores:
        # step_scores shape: (1, vocab_size)
        probs     = torch.softmax(step_scores[0], dim=-1)
        top_prob  = probs.max().item()
        token_probs.append(top_prob)
 
    if not token_probs:
        return {"mean_prob": 0.0, "entropy": 1.0, "confidence": 0.0, "token_count": 0}
 
    token_probs    = np.array(token_probs)
    mean_prob      = float(np.mean(token_probs))
 
    # Entropy over generated tokens (lower = more confident)
    # Clip to avoid log(0)
    clipped        = np.clip(token_probs, 1e-10, 1.0)
    entropy        = float(-np.mean(clipped * np.log(clipped)))
 
    # Normalise entropy to [0,1] confidence
    # Max entropy for a token ~ log(vocab_size) ≈ 11 for 60k vocab
    max_entropy    = np.log(60000)
    confidence     = float(1.0 - (entropy / max_entropy))
    confidence     = max(0.0, min(1.0, confidence))
 
    return {
        "mean_token_prob": round(mean_prob,  4),
        "entropy":         round(entropy,    4),
        "confidence":      round(confidence, 4),
        "token_count":     len(token_probs),
    }
 
 
def batch_compute_confidence(outputs, tokenizer, model, device, sample_n=20):
    """
    Computes confidence for a sample of outputs (full set is slow).
    Returns per-example confidence dicts and mean confidence.
    """
    from src.generator import STRATEGIES
 
    strat       = STRATEGIES["structured"]
    sys_prompt  = strat["system"]
    prompt_fn   = lambda q, c: strat["template"].format(context=c, query=q)
 
    # Sample to keep runtime manageable
    sample      = outputs[:sample_n]
    conf_scores = []
 
    print(f"Computing confidence for {len(sample)} samples...")
 
    for i, output in enumerate(sample, 1):
        # Re-retrieve context for this query
        retrieved = retriever_.retrieve(output["question"], initial_k=20, final_k=5)
        context   = build_context(retrieved, max_words=600)
 
        conf = compute_confidence(
            query          = output["question"],
            context        = context,
            answer         = output["pred_answer"],
            system_prompt  = sys_prompt,
            user_prompt_fn = prompt_fn,
            tokenizer      = generator_.tokenizer,
            model          = generator_.model,
            device         = device,
            max_new_tokens = 150,
        )
        conf_scores.append(conf)
 
        if i % 5 == 0:
            print(f"  {i}/{len(sample)} done — "
                  f"avg confidence so far: {np.mean([c['confidence'] for c in conf_scores]):.3f}")
 
    mean_conf = float(np.mean([c["confidence"] for c in conf_scores]))
    return conf_scores, mean_conf
 

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — All 10 evaluation metrics
# ─────────────────────────────────────────────────────────────────────────────
 
def compute_all_metrics(outputs, source_lookup, k=5):
    """
    Computes all 10 evaluation metrics for a list of inference outputs.
 
    Returns:
        dict of metric_name -> mean_score
        dict of metric_name -> list of per-query scores
    """
    predictions = [o["pred_answer"]  for o in outputs]
    references  = [o["gold_answer"]  for o in outputs]
    questions   = [o["question"]     for o in outputs]
 
    per_query = defaultdict(list)
 
    # ── 1. ROUGE-1 ────────────────────────────────────────────────────────────
    rouge_sc = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )
    for pred, ref in zip(predictions, references):
        r = rouge_sc.score(ref, pred)
        per_query["rouge1"].append(r["rouge1"].fmeasure)
        per_query["rouge2"].append(r["rouge2"].fmeasure)
        per_query["rougeL"].append(r["rougeL"].fmeasure)
 
    # ── 2. BERTScore ──────────────────────────────────────────────────────────
    print("  Computing BERTScore...")
    _, _, F1 = bert_score_fn(
        predictions, references,
        model_type="distilbert-base-uncased",
        lang="en", verbose=False,
    )
    per_query["bertscore"] = F1.numpy().tolist()
 
    # ── 3. METEOR ─────────────────────────────────────────────────────────────
    print("  Computing METEOR...")
    for pred, ref in zip(predictions, references):
        try:
            score = meteor_score(
                [ref.lower().split()],
                pred.lower().split()
            )
        except Exception:
            score = 0.0
        per_query["meteor"].append(float(score))
 
    # ── 4. Answer F1 ──────────────────────────────────────────────────────────
    def token_f1(pred, ref):
        pred_toks = set(pred.lower().split())
        ref_toks  = set(ref.lower().split())
        common    = pred_toks & ref_toks
        if not common:
            return 0.0
        p = len(common) / len(pred_toks)
        r = len(common) / len(ref_toks)
        return 2 * p * r / (p + r)
 
    for pred, ref in zip(predictions, references):
        per_query["answer_f1"].append(token_f1(pred, ref))
 
    # ── 5. Exact Match ────────────────────────────────────────────────────────
    def normalise(text):
        text = text.lower().strip()
        text = re.sub(r'[^\w\s]', '', text)
        return re.sub(r'\s+', ' ', text)
 
    for pred, ref in zip(predictions, references):
        per_query["exact_match"].append(
            1.0 if normalise(pred) == normalise(ref) else 0.0
        )
 
    # ── 6. Answer Relevance ───────────────────────────────────────────────────
    print("  Computing answer relevance...")
    q_embs = embedder_.encode_documents(questions, show_progress=False)
    a_embs = embedder_.encode_documents(predictions, show_progress=False)
    for q_emb, a_emb in zip(q_embs, a_embs):
        sim = sklearn_cosine(q_emb.reshape(1,-1), a_emb.reshape(1,-1))[0][0]
        per_query["answer_relevance"].append(float(sim))
 
    # ── 7. Faithfulness ───────────────────────────────────────────────────────
    def normalise_set(text):
        return set(re.sub(r'[^\w\s]', '', text.lower()).split())
 
    for output in outputs:
        pred_words    = normalise_set(output.get("pred_answer", ""))
        context_words = set()
        for chunk in output.get("retrieved", []):
            context_words.update(normalise_set(chunk.get("text", "")))
        if not pred_words or not context_words:
            per_query["faithfulness"].append(0.0)
            continue
        overlap = pred_words & context_words
        per_query["faithfulness"].append(len(overlap) / len(pred_words))
 
    # ── 8. Context Precision ─────────────────────────────────────────────────
    for output in outputs:
        gold       = source_lookup.get(output["question"], "").lower()
        retrieved  = output.get("retrieved", [])
        if not gold or not retrieved:
            per_query["context_precision"].append(0.0)
            continue
        relevant = sum(
            1 for c in retrieved
            if gold in c.get("doc_title", "").lower()
            or c.get("doc_title", "").lower() in gold
        )
        per_query["context_precision"].append(relevant / len(retrieved))
 
    # ── 9. MRR@5 and 10. NDCG@5 ──────────────────────────────────────────────
    for output in outputs:
        gold      = source_lookup.get(output["question"], "").lower()
        retrieved = output.get("retrieved", [])
        if not gold or not retrieved:
            per_query["mrr5"].append(0.0)
            per_query["ndcg5"].append(0.0)
            continue
 
        relevance = [
            1 if (gold in c.get("doc_title","").lower()
                  or c.get("doc_title","").lower() in gold) else 0
            for c in retrieved[:k]
        ]
 
        # MRR@5
        mrr = 0.0
        for rank, rel in enumerate(relevance, 1):
            if rel == 1:
                mrr = 1.0 / rank
                break
        per_query["mrr5"].append(mrr)
 
        # NDCG@5
        dcg  = sum(r / np.log2(i+1) for i, r in enumerate(relevance, 1))
        idcg = sum(r / np.log2(i+1)
                   for i, r in enumerate(sorted(relevance, reverse=True), 1))
        per_query["ndcg5"].append(dcg / idcg if idcg > 0 else 0.0)
 
    # Compute means
    means = {
        metric: float(np.mean(scores)) if scores else 0.0
        for metric, scores in per_query.items()
    }
 
    return means, dict(per_query)
 

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Ablation configurations
# Each config removes or changes one component to show its contribution
# ─────────────────────────────────────────────────────────────────────────────
 
ABLATION_CONFIGS = {
 
    "Full system (ours)": {
        "retrieval":  "hybrid",
        "rerank":     True,
        "chunking":   "semantic",
        "prompting":  "structured",
        "description":"Complete pipeline — hybrid retrieval + reranking + structured prompting",
    },
    "No reranker": {
        "retrieval":  "hybrid",
        "rerank":     False,
        "chunking":   "semantic",
        "prompting":  "structured",
        "description":"Remove cross-encoder reranker — use RRF order directly",
    },
    "Dense only (no BM25)": {
        "retrieval":  "dense",
        "rerank":     True,
        "chunking":   "semantic",
        "prompting":  "structured",
        "description":"Remove BM25 — use only dense FAISS retrieval",
    },
    "BM25 only (no dense)": {
        "retrieval":  "bm25",
        "rerank":     True,
        "chunking":   "semantic",
        "prompting":  "structured",
        "description":"Remove dense retrieval — use only BM25 keyword search",
    },
    "Fixed chunking (256w)": {
        "retrieval":  "hybrid",
        "rerank":     True,
        "chunking":   "fixed",
        "prompting":  "structured",
        "description":"Replace semantic chunking with fixed 256-word chunks",
    },
    "Zero-shot prompting": {
        "retrieval":  "hybrid",
        "rerank":     True,
        "chunking":   "semantic",
        "prompting":  "zero_shot",
        "description":"Replace structured prompting with zero-shot",
    },
    "No RAG (baseline)": {
        "retrieval":  "none",
        "rerank":     False,
        "chunking":   "none",
        "prompting":  "zero_shot",
        "description":"LLM with no retrieved context — pure generation baseline",
    },
}
 
 
def run_ablation_config(config_name, config, train_set, max_queries=None):
    """
    Runs inference for one ablation configuration.
    Returns list of output dicts compatible with compute_all_metrics.
    """
    queries = train_set[:max_queries] if max_queries else train_set
    outputs = []
 
    print(f"\n  Running: {config_name}")
    print(f"  {config['description']}")
 
    # Load appropriate chunks for chunking strategy
    if config["chunking"] == "fixed":
        try:
            with open("data/processed/chunks_fixed_256.json", encoding="utf-8") as f:
                alt_chunks = json.load(f)
            # Build a temporary BM25 for fixed chunks
            from rank_bm25 import BM25Okapi
            alt_bm25 = BM25Okapi([c["text"].lower().split() for c in alt_chunks])
            # Build temporary FAISS index for fixed chunks
            alt_texts = [c["text"] for c in alt_chunks]
            alt_embs  = embedder_.encode_documents(alt_texts, show_progress=False)
            alt_index = faiss.IndexFlatIP(alt_embs.shape[1])
            alt_index.add(alt_embs.astype(np.float32))
            use_alt_chunks = True
        except FileNotFoundError:
            print("  Fixed chunks not found — using semantic chunks")
            use_alt_chunks = False
    else:
        use_alt_chunks = False
 
    for i, qa in enumerate(queries, 1):
        question = qa["question"]
        gold     = qa["answer"]
 
        # ── Retrieval ────────────────────────────────────────────────────────
        if config["retrieval"] == "none":
            retrieved = []
            context   = ""
 
        elif config["retrieval"] == "hybrid":
            if use_alt_chunks:
                # Dense retrieval with alt chunks
                q_emb = embedder_.encode_query(question)
                scores, indices = alt_index.search(q_emb, 10)
                dense_r = [{"chunk": alt_chunks[idx], "score": float(s),
                             "method": "dense", "index": int(idx)}
                           for s, idx in zip(scores[0], indices[0]) if idx != -1]
                # BM25 with alt chunks
                bm25_scores = alt_bm25.get_scores(question.lower().split())
                top_bm25    = np.argsort(bm25_scores)[::-1][:10]
                bm25_r = [{"chunk": alt_chunks[idx], "score": float(bm25_scores[idx]),
                            "method": "bm25", "index": int(idx)}
                          for idx in top_bm25 if bm25_scores[idx] > 0]
                fused = retriever_.reciprocal_rank_fusion(dense_r, bm25_r)
                retrieved = (retriever_.rerank(question, fused[:20], top_n=5)
                             if config["rerank"] else fused[:5])
            else:
                if config["rerank"]:
                    retrieved = retriever_.retrieve(question, initial_k=20, final_k=5)
                else:
                    dense_r = retriever_.dense_retrieve(question, k=20)
                    bm25_r  = retriever_.bm25_retrieve(question,  k=20)
                    fused   = retriever_.reciprocal_rank_fusion(dense_r, bm25_r)
                    retrieved = fused[:5]
            context = build_context(retrieved, max_words=600)
 
        elif config["retrieval"] == "dense":
            dense_r   = retriever_.dense_retrieve(question, k=20)
            retrieved = (retriever_.rerank(question, dense_r, top_n=5)
                         if config["rerank"] else dense_r[:5])
            context   = build_context(retrieved, max_words=600)
 
        elif config["retrieval"] == "bm25":
            bm25_r    = retriever_.bm25_retrieve(question, k=20)
            retrieved = (retriever_.rerank(question, bm25_r, top_n=5)
                         if config["rerank"] else bm25_r[:5])
            context   = build_context(retrieved, max_words=600)
 
        # ── Generation ───────────────────────────────────────────────────────
        if config["retrieval"] == "none":
            # No-context baseline
            from src.generator import STRATEGIES
            strat      = STRATEGIES["zero_shot"]
            sys_prompt = strat["system"]
            user_msg   = f"Question: {question}\n\nAnswer:"
            from transformers import AutoTokenizer
            messages   = [
                {"role": "system", "content": sys_prompt},
                {"role": "user",   "content": user_msg},
            ]
            text   = generator_.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = generator_.tokenizer([text], return_tensors="pt").to(device)
            with torch.no_grad():
                out = generator_.model.generate(
                    **inputs, max_new_tokens=200, temperature=0.3,
                    do_sample=True, pad_token_id=generator_.tokenizer.eos_token_id,
                )
            gen   = out[0][inputs["input_ids"].shape[1]:]
            answer= generator_.tokenizer.decode(gen, skip_special_tokens=True).strip()
        else:
            answer = generator_.generate(
                query=question, context=context,
                strategy=config["prompting"],
                max_new_tokens=200, temperature=0.3,
            )
 
        outputs.append({
            "id":           qa.get("id", f"Q{i}"),
            "question":     question,
            "gold_answer":  gold,
            "pred_answer":  answer,
            "type":         qa.get("type", "unknown"),
            "difficulty":   qa.get("difficulty", "unknown"),
            "retrieved":    [
                {"doc_title": r["chunk"].get("doc_title",""),
                 "text":      r["chunk"].get("text","")[:200]}
                for r in retrieved
            ],
        })
 
        if i % 10 == 0:
            print(f"    {i}/{len(queries)} queries done")
 
    return outputs
 

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Run all ablation configurations
# This is the main loop — takes 20-40 minutes total
# ─────────────────────────────────────────────────────────────────────────────
 
# Use a subset for speed — set to None to run all 92 train queries
# 30 queries gives reliable enough estimates for the ablation table
MAX_QUERIES = 30
 
print("="*65)
print("  ABLATION STUDY")
print(f"  Running {len(ABLATION_CONFIGS)} configurations × {MAX_QUERIES} queries")
print("="*65)
 
ablation_results  = {}   # config_name -> {metric -> score}
ablation_outputs  = {}   # config_name -> list of output dicts
ablation_per_query= {}   # config_name -> {metric -> list}
 
for config_name, config in ABLATION_CONFIGS.items():
 
    # Check if we already have the full system results from notebook 04
    if config_name == "Full system (ours)" and len(train_outputs) >= MAX_QUERIES:
        print(f"\n  Using existing outputs for: {config_name}")
        outputs = train_outputs[:MAX_QUERIES]
    else:
        outputs = run_ablation_config(config_name, config, train_set, MAX_QUERIES)
 
    ablation_outputs[config_name] = outputs
 
    # Compute all 10 metrics
    print(f"  Computing metrics for: {config_name}...")
    means, per_query = compute_all_metrics(outputs, source_lookup)
    ablation_results[config_name]   = means
    ablation_per_query[config_name] = per_query
 
    # Save outputs
    safe_name = re.sub(r'[^\w]', '_', config_name)
    with open(f"outputs/ablation/{safe_name}.json", "w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)
 
    print(f"  Done — ROUGE-L={means['rougeL']:.4f}  "
          f"BERT={means['bertscore']:.4f}  "
          f"NDCG@5={means['ndcg5']:.4f}")
 
print("\nAll ablation configurations complete")
 

  ABLATION STUDY
  Running 7 configurations × 30 queries

  Using existing outputs for: Full system (ours)
  Computing metrics for: Full system (ours)...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing METEOR...
  Computing answer relevance...
  Done — ROUGE-L=0.3533  BERT=0.8381  NDCG@5=0.9276

  Running: No reranker
  Remove cross-encoder reranker — use RRF order directly
    10/30 queries done
    20/30 queries done
    30/30 queries done
  Computing metrics for: No reranker...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing METEOR...
  Computing answer relevance...
  Done — ROUGE-L=0.2193  BERT=0.7946  NDCG@5=0.9276

  Running: No RAG (baseline)
  LLM with no retrieved context — pure generation baseline
    10/30 queries done
    20/30 queries done
    30/30 queries done
  Computing metrics for: No RAG (baseline)...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing METEOR...
  Computing answer relevance...
  Done — ROUGE-L=0.1495  BERT=0.7683  NDCG@5=0.0000

All ablation configurations complete


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Compute confidence scores for full system
# ─────────────────────────────────────────────────────────────────────────────
 
print("\nComputing confidence scores for full system...")
 
full_outputs  = ablation_outputs["Full system (ours)"]
conf_scores, mean_conf = batch_compute_confidence(
    full_outputs,
    tokenizer  = generator_.tokenizer,
    model      = generator_.model,
    device     = device,
    sample_n   = min(20, len(full_outputs)),
)
 
print(f"\nConfidence results (n={len(conf_scores)}):")
print(f"  Mean confidence  : {mean_conf:.4f}")
print(f"  Mean token prob  : {np.mean([c['mean_token_prob'] for c in conf_scores]):.4f}")
print(f"  Mean entropy     : {np.mean([c['entropy'] for c in conf_scores]):.4f}")
print(f"  High conf (>0.7) : {sum(1 for c in conf_scores if c['confidence']>0.7)}/{len(conf_scores)}")
print(f"  Low conf  (<0.3) : {sum(1 for c in conf_scores if c['confidence']<0.3)}/{len(conf_scores)}")
 
# Attach confidence to full system outputs
for i, output in enumerate(full_outputs):
    if i < len(conf_scores):
        output["confidence"] = conf_scores[i]
    else:
        output["confidence"] = {
            "mean_token_prob": 0.0,
            "entropy": 0.0,
            "confidence": mean_conf,
            "token_count": 0,
        }
 


Computing confidence scores for full system...
Computing confidence for 20 samples...
  5/20 done — avg confidence so far: 0.998
  10/20 done — avg confidence so far: 0.997
  15/20 done — avg confidence so far: 0.997
  20/20 done — avg confidence so far: 0.997

Confidence results (n=20):
  Mean confidence  : 0.9969
  Mean token prob  : 0.9568
  Mean entropy     : 0.0337
  High conf (>0.7) : 20/20
  Low conf  (<0.3) : 0/20


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Statistical significance tests
# Compare each ablation against the full system using paired t-tests
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*65)
print("  STATISTICAL SIGNIFICANCE TESTS")
print("  (Full system vs each ablation — paired t-test on ROUGE-L)")
print("="*65)
 
full_rougeL = ablation_per_query["Full system (ours)"]["rougeL"]
sig_results = {}
 
for config_name in ABLATION_CONFIGS:
    if config_name == "Full system (ours)":
        continue
 
    other_rougeL = ablation_per_query[config_name]["rougeL"]
    n = min(len(full_rougeL), len(other_rougeL))
 
    if n < 5:
        continue
 
    t_stat, p_val = stats.ttest_rel(full_rougeL[:n], other_rougeL[:n])
    sig = p_val < 0.05
    diff = np.mean(full_rougeL[:n]) - np.mean(other_rougeL[:n])
 
    sig_results[config_name] = {
        "t_stat":      round(float(t_stat),  4),
        "p_value":     round(float(p_val),   4),
        "significant": sig,
        "rougeL_diff": round(float(diff),    4),
    }
 
    sign = "+" if diff >= 0 else ""
    print(f"\n  Full system vs {config_name}")
    print(f"    ROUGE-L diff : {sign}{diff:.4f}")
    print(f"    t={t_stat:.3f}  p={p_val:.4f}  "
          f"{'SIGNIFICANT ✓' if sig else 'not significant'}")
 


  STATISTICAL SIGNIFICANCE TESTS
  (Full system vs each ablation — paired t-test on ROUGE-L)

  Full system vs No reranker
    ROUGE-L diff : +0.0611
    t=1.267  p=0.2151  not significant

  Full system vs Dense only (no BM25)
    ROUGE-L diff : +0.0362
    t=0.892  p=0.3795  not significant

  Full system vs BM25 only (no dense)
    ROUGE-L diff : +0.0727
    t=1.915  p=0.0654  not significant

  Full system vs Fixed chunking (256w)
    ROUGE-L diff : +0.1922
    t=3.784  p=0.0007  SIGNIFICANT ✓

  Full system vs Zero-shot prompting
    ROUGE-L diff : +0.1340
    t=3.487  p=0.0016  SIGNIFICANT ✓

  Full system vs No RAG (baseline)
    ROUGE-L diff : +0.2038
    t=4.355  p=0.0002  SIGNIFICANT ✓


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Main results table (all 10 metrics × all configurations)
# This is the primary output for your presentation
# ─────────────────────────────────────────────────────────────────────────────
 
METRICS_DISPLAY = [
    ("rouge1",           "ROUGE-1"),
    ("rouge2",           "ROUGE-2"),
    ("rougeL",           "ROUGE-L"),
    ("bertscore",        "BERTScore F1"),
    ("meteor",           "METEOR"),
    ("answer_f1",        "Answer F1"),
    ("exact_match",      "Exact Match"),
    ("answer_relevance", "Answer Relevance"),
    ("faithfulness",     "Faithfulness"),
    ("mrr5",             "MRR@5"),
    ("ndcg5",            "NDCG@5"),
    ("context_precision","Context Precision"),
]
 
# Print table
col_w = 22
print("\n" + "="*(col_w + len(ABLATION_CONFIGS)*12))
print("  ABLATION STUDY — COMPLETE RESULTS TABLE (all metrics)")
print("="*(col_w + len(ABLATION_CONFIGS)*12))
 
# Header
header = f"  {'Metric':<{col_w}}"
for name in ABLATION_CONFIGS:
    short = name[:10]
    header += f" {short:>11}"
print(header)
print("  " + "─"*(col_w + len(ABLATION_CONFIGS)*12))
 
# Rows
for metric_key, metric_label in METRICS_DISPLAY:
    row = f"  {metric_label:<{col_w}}"
    best_val = max(
        ablation_results[c].get(metric_key, 0)
        for c in ABLATION_CONFIGS
    )
    for config_name in ABLATION_CONFIGS:
        val = ablation_results[config_name].get(metric_key, 0)
        mark = "*" if abs(val - best_val) < 0.001 else " "
        row += f" {val:>10.4f}{mark}"
    print(row)
 
print("  " + "─"*(col_w + len(ABLATION_CONFIGS)*12))
print("  * = best result for this metric")
print(f"\n  Confidence (full system): {mean_conf:.4f} mean")
 


  ABLATION STUDY — COMPLETE RESULTS TABLE (all metrics)
  Metric                  Full syste  No reranke  Dense only  BM25 only   Fixed chun  Zero-shot   No RAG (ba
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
  ROUGE-1                    0.4005*     0.3441      0.3634      0.3402      0.2180      0.2764      0.2042 
  ROUGE-2                    0.2404*     0.1807      0.1924      0.1610      0.0529      0.1123      0.0511 
  ROUGE-L                    0.3533*     0.2921      0.3171      0.2805      0.1611      0.2193      0.1495 
  BERTScore F1               0.8381*     0.8194      0.8237      0.8200      0.7874      0.7946      0.7683 
  METEOR                     0.3354*     0.2804      0.2933      0.2467      0.1500      0.2342      0.1989 
  Answer F1                  0.3561*     0.3037      0.3298      0.2846      0.1949      0.2267      0.1773 
  Exact Match                0.0000*     0.0000*     0.0000*     0.0000

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Per-question-type breakdown for full system
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*65)
print("  FULL SYSTEM — RESULTS BY QUESTION TYPE")
print("="*65)
 
full_outputs_all = ablation_outputs["Full system (ours)"]
type_groups      = defaultdict(list)
diff_groups      = defaultdict(list)
 
for i, output in enumerate(full_outputs_all):
    qtype = output.get("type", "unknown")
    qdiff = output.get("difficulty", "unknown")
    rl    = ablation_per_query["Full system (ours)"]["rougeL"][i] \
            if i < len(ablation_per_query["Full system (ours)"]["rougeL"]) else 0
    bs    = ablation_per_query["Full system (ours)"]["bertscore"][i] \
            if i < len(ablation_per_query["Full system (ours)"]["bertscore"]) else 0
    mt    = ablation_per_query["Full system (ours)"]["meteor"][i] \
            if i < len(ablation_per_query["Full system (ours)"]["meteor"]) else 0
    type_groups[qtype].append((rl, bs, mt))
    diff_groups[qdiff].append((rl, bs, mt))
 
print(f"\n  {'Type':<16} {'N':>4} {'ROUGE-L':>9} {'BERTScore':>10} {'METEOR':>8}")
print("  " + "─"*52)
for qtype, vals in sorted(type_groups.items()):
    rls = [v[0] for v in vals]
    bss = [v[1] for v in vals]
    mts = [v[2] for v in vals]
    print(f"  {qtype:<16} {len(vals):>4} {np.mean(rls):>9.4f} "
          f"{np.mean(bss):>10.4f} {np.mean(mts):>8.4f}")
 
print(f"\n  {'Difficulty':<12} {'N':>4} {'ROUGE-L':>9} {'BERTScore':>10} {'METEOR':>8}")
print("  " + "─"*48)
for qdiff, vals in sorted(diff_groups.items()):
    rls = [v[0] for v in vals]
    bss = [v[1] for v in vals]
    mts = [v[2] for v in vals]
    print(f"  {qdiff:<12} {len(vals):>4} {np.mean(rls):>9.4f} "
          f"{np.mean(bss):>10.4f} {np.mean(mts):>8.4f}")
 


  FULL SYSTEM — RESULTS BY QUESTION TYPE

  Type                N   ROUGE-L  BERTScore   METEOR
  ────────────────────────────────────────────────────
  comparative         1    0.2353     0.8426   0.1363
  cultural           12    0.3766     0.8387   0.3449
  factual             8    0.4072     0.8551   0.4250
  ingredient          2    0.4030     0.8513   0.3302
  procedural          7    0.2543     0.8131   0.2465

  Difficulty      N   ROUGE-L  BERTScore   METEOR
  ────────────────────────────────────────────────
  easy            6    0.3541     0.8261   0.3240
  hard           14    0.4252     0.8620   0.3968
  medium         10    0.2521     0.8118   0.2561


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Best and worst predictions with confidence
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*65)
print("  QUALITATIVE ANALYSIS — BEST AND WORST PREDICTIONS")
print("="*65)
 
full_rougeL_scores = ablation_per_query["Full system (ours)"]["rougeL"]
ranked = sorted(
    zip(full_outputs_all, full_rougeL_scores),
    key=lambda x: x[1], reverse=True
)
 
print("\nTOP 3 BEST PREDICTIONS:")
print("─"*65)
for output, rl in ranked[:3]:
    conf = output.get("confidence", {}).get("confidence", "N/A")
    print(f"\n  Q    : {output['question']}")
    print(f"  Gold : {output['gold_answer'][:100]}...")
    print(f"  Pred : {output['pred_answer'][:100]}...")
    print(f"  ROUGE-L={rl:.3f}  Confidence={conf}")
 
print("\nBOTTOM 3 WORST PREDICTIONS (error analysis):")
print("─"*65)
for output, rl in ranked[-3:]:
    conf = output.get("confidence", {}).get("confidence", "N/A")
    print(f"\n  Q    : {output['question']}")
    print(f"  Gold : {output['gold_answer'][:100]}...")
    print(f"  Pred : {output['pred_answer'][:100]}...")
    print(f"  ROUGE-L={rl:.3f}  Confidence={conf}")
    # Diagnose why it failed
    if rl < 0.1:
        print(f"  WHY  : Low overlap — model likely hallucinated or answer is paraphrased")
    elif rl < 0.2:
        print(f"  WHY  : Partial match — answer captures some facts but misses key details")
 


  QUALITATIVE ANALYSIS — BEST AND WORST PREDICTIONS

TOP 3 BEST PREDICTIONS:
─────────────────────────────────────────────────────────────────

  Q    : What is the historical origin of deep frying?
  Gold : Early records and cookbooks suggest that the practice began in certain European countries before oth...
  Pred : Early records and cookbooks suggest that the practice of deep frying originated in certain European ...
  ROUGE-L=0.878  Confidence=0.9991

  Q    : How did the world's earliest eating establishments recognize restaurants in the modern sense?
  Gold : Street food became an integral aspect of Chinese food culture in the 7th century during the Tang dyn...
  Pred : Street food became an integral aspect of Chinese food culture in the Song dynasty during the 11th an...
  ROUGE-L=0.846  Confidence=0.9991

  Q    : What are some common fillings used in spring rolls in Southeast Asia?
  Gold : Common fillings used in spring rolls in Southeast Asia include vegetables such as car

In [13]:
# =============================================================================
# CELL 12 — Save complete ablation report (FIXED)
# Converts all numpy types to native Python before JSON serialisation
# =============================================================================

import numpy as np
from datetime import datetime

def make_serialisable(obj):
    """
    Recursively converts numpy types to native Python types
    so json.dump never throws a TypeError.
    """
    if isinstance(obj, dict):
        return {k: make_serialisable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serialisable(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


ablation_report = {
    "created_at":      datetime.now().isoformat(),
    "num_queries":     MAX_QUERIES,
    "configurations":  list(ABLATION_CONFIGS.keys()),
    "metrics_computed": 12,

    "results": {
        config_name: {
            metric_key: float(ablation_results[config_name].get(metric_key, 0))
            for metric_key, _ in METRICS_DISPLAY
        }
        for config_name in ABLATION_CONFIGS
    },

    "confidence": {
        "mean_confidence":  float(mean_conf),
        "mean_token_prob":  float(np.mean([c["mean_token_prob"] for c in conf_scores])),
        "mean_entropy":     float(np.mean([c["entropy"]         for c in conf_scores])),
        "high_conf_rate":   float(sum(1 for c in conf_scores if c["confidence"] > 0.7)
                                  / max(len(conf_scores), 1)),
        "per_example": [
            {k: float(v) if isinstance(v, (int, float, np.floating, np.integer))
                         else bool(v) if isinstance(v, (bool, np.bool_))
                         else v
             for k, v in c.items()}
            for c in conf_scores
        ],
    },

    "significance_tests": {
        config_name: {
            "t_stat":      float(res["t_stat"]),
            "p_value":     float(res["p_value"]),
            "significant": bool(res["significant"]),
            "rougeL_diff": float(res["rougeL_diff"]),
        }
        for config_name, res in sig_results.items()
    },

    "best_config": max(
        ablation_results.items(),
        key=lambda x: x[1].get("rougeL", 0)
    )[0],
}

# Apply full serialisation safety pass
ablation_report = make_serialisable(ablation_report)

with open("outputs/ablation/ablation_report.json", "w", encoding="utf-8") as f:
    json.dump(ablation_report, f, ensure_ascii=False, indent=2)

print("\n" + "="*65)
print("  ABLATION STUDY COMPLETE")
print("="*65)
print(f"\n  Configurations tested  : {len(ABLATION_CONFIGS)}")
print(f"  Metrics computed       : 12")
print(f"  Queries per config     : {MAX_QUERIES}")
print(f"  Best configuration     : {ablation_report['best_config']}")
print(f"  Mean confidence        : {mean_conf:.4f}")

print(f"\n  Saved:")
print(f"    outputs/ablation/ablation_report.json")
print(f"    outputs/ablation/<config_name>.json (per config)")

print(f"\n  Key findings for presentation:")
full_rougeL = ablation_results["Full system (ours)"].get("rougeL", 0)
for config_name, res in ablation_results.items():
    if config_name == "Full system (ours)":
        continue
    diff = full_rougeL - res.get("rougeL", 0)
    sig  = sig_results.get(config_name, {})
    sig_str = "✓ significant" if sig.get("significant") else ""
    if diff > 0:
        print(f"    Removing {config_name:<32} costs {diff:+.4f} ROUGE-L  {sig_str}")

print(f"\n  Confidence interpretation:")
print(f"    Score of {mean_conf:.4f} means the model is highly certain")
print(f"    about its token choices — typical for small instruction-tuned models.")
print(f"    Low entropy ({np.mean([c['entropy'] for c in conf_scores]):.4f}) confirms")
print(f"    the model is not randomly sampling but following learned patterns.")

print(f"\n  completed and saved.")
print("="*65)


  ABLATION STUDY COMPLETE

  Configurations tested  : 7
  Metrics computed       : 12
  Queries per config     : 30
  Best configuration     : Full system (ours)
  Mean confidence        : 0.9969

  Saved:
    outputs/ablation/ablation_report.json
    outputs/ablation/<config_name>.json (per config)

  Key findings for presentation:
    Removing No reranker                      costs +0.0611 ROUGE-L  
    Removing Dense only (no BM25)             costs +0.0362 ROUGE-L  
    Removing BM25 only (no dense)             costs +0.0727 ROUGE-L  
    Removing Fixed chunking (256w)            costs +0.1922 ROUGE-L  ✓ significant
    Removing Zero-shot prompting              costs +0.1340 ROUGE-L  ✓ significant
    Removing No RAG (baseline)                costs +0.2038 ROUGE-L  ✓ significant

  Confidence interpretation:
    Score of 0.9969 means the model is highly certain
    about its token choices — typical for small instruction-tuned models.
    Low entropy (0.0337) confirms
    the model